# Cortical development and ERBB4 perturbation

Paired RNA and ATAC measurements are used to examine human cortical development. The analysis follows cell-state transitions and compares the predicted effects of ERBB4 perturbation on trajectories and terminal cell-type composition (Figure 6).


In [ ]:
from pathlib import Path
import json

from mofi_notebook import setup_notebook, suppress_live_plot_output

ROOT, DEVICE, _ = setup_notebook()
print(f"Device: {DEVICE}")


## Continuous-time dynamics


In [ ]:
from CytoBridge.reproducibility import require_paper_data, run_real_downstream

DATASET = "hc_author5"
paths = require_paper_data(DATASET, ROOT)
paper_output = ROOT / "results" / "figure_6_paper"

with suppress_live_plot_output():
    downstream = run_real_downstream(
        DATASET,
        output_dir=paper_output,
        selected_figures=[],
        device=DEVICE,
        sample_max_cells=None,
        reuse_cache=True,
        repo_root=ROOT,
    )
run_dir = Path(downstream["run_dir"])
run_dir


## Observed and interpolated cell states

Time slices compare measured populations with states predicted between observations.


In [ ]:
from CytoBridge.manuscript import render_hc_time_slices
from CytoBridge.reproducibility import write_resolved_analysis_config

local_cfg = write_resolved_analysis_config(
    DATASET,
    output_dir=paper_output,
    config_path=ROOT / "results" / "figure6_resolved.yaml",
    device=DEVICE,
    repo_root=ROOT,
)
with suppress_live_plot_output():
    time_slice_dir = render_hc_time_slices(
        config_path=local_cfg,
        output_dir=run_dir / "figures" / "time_slices",
        repo_root=ROOT,
    )
time_slice_dir


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "time_slices" / "author5_time_slice_panels_umap.png")


## Lineage transitions

Sankey links summarize model trajectories assigned to cell types at consecutive time points.


In [ ]:
from CytoBridge.manuscript import render_hc_lineage

with suppress_live_plot_output():
    lineage_dir = render_hc_lineage(run_dir=run_dir, repo_root=ROOT)
lineage_dir


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "author5_lineage" / "author5_lineage_sankey.png")


## ERBB4 perturbation

The same fitted dynamics and cross-modality map are used across perturbation conditions.


In [ ]:
import shutil
from CytoBridge.pl.tf.context import TFRunContext

ctx = TFRunContext.from_run_dir(run_dir)
trajectory_index_csv = ROOT / "configs" / "analysis_inputs" / "hc_erbb4_paper6_indices.csv"
fate_index_csv = ROOT / "configs" / "analysis_inputs" / "hc_erbb4_indices.csv"
erbb4_dir = run_dir / "figures" / "tf" / "erbb4"
trajectory_dir = erbb4_dir / "trajectory"
trajectory_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    ROOT / "paper_data" / "umap" / DATASET / "erbb4_primary.pkl",
    trajectory_dir / "umap_model_primary.pkl",
)
shutil.copy2(
    ROOT / "paper_data" / "umap" / DATASET / "erbb4_secondary.pkl",
    trajectory_dir / "umap_model_secondary.pkl",
)


## Perturbed trajectories

ERBB4 perturbations are simulated at seven scores. The panels compare the unperturbed condition with the strongest positive perturbation in RNA and ATAC space.


In [ ]:
from CytoBridge.pl.tf.perturb import plot_perturbation_trajectories

with suppress_live_plot_output():
    trajectories = plot_perturbation_trajectories(
        ctx,
        indices_csv=str(trajectory_index_csv),
        scores=[-9, -6, -3, 0, 3, 6, 9],
        sample_n=6,
        seed=4,
        device=DEVICE,
        output_dir=str(trajectory_dir),
        dim_reduction="umap",
    )
trajectories


### RNA trajectories: unperturbed ERBB4


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "tf" / "erbb4" / "trajectory" / "perturb_index_traj_primary__g-mge-erbb4__z-0__dr-umap.png")


### RNA trajectories: increased ERBB4


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "tf" / "erbb4" / "trajectory" / "perturb_index_traj_primary__g-mge-erbb4__z-9__dr-umap.png")


### ATAC trajectories: unperturbed ERBB4


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "tf" / "erbb4" / "trajectory" / "perturb_index_traj_secondary__g-mge-erbb4__z-0__dr-umap.png")


### ATAC trajectories: increased ERBB4


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "tf" / "erbb4" / "trajectory" / "perturb_index_traj_secondary__g-mge-erbb4__z-9__dr-umap.png")


## Terminal fate composition

The heatmap shows predicted cell-type probabilities at week 52 across ERBB4 perturbation scores.


In [ ]:
from CytoBridge.pl.tf.perturb import build_perturbation_fate_sweep

with suppress_live_plot_output():
    fate = build_perturbation_fate_sweep(
        ctx,
        indices_csv=str(fate_index_csv),
        target_group="mge_erbb4",
        target_genes=["ERBB4"],
        scores=[-9, -6, -3, 0, 3, 6, 9],
        sample_n="all",
        seed=4,
        device=DEVICE,
        output_table_dir=str(run_dir / "tables" / "tf" / "erbb4_fate"),
        output_figure_dir=str(erbb4_dir / "fate"),
        sweep_cell_types=["RG", "IPC", "IN-fetal", "IN-CGE", "IN-MGE"],
        focus_curve_cell_types=["RG", "IPC", "IN-fetal", "IN-CGE", "IN-MGE"],
        focus_curve_scores=[-9, -6, -3, 0, 3, 6, 9],
        heatmap_z_order=[-9, -6, -3, 0, 3, 6, 9],
        heatmap_orientation="z_by_cell_type",
        heatmap_cell_type_order=["IN-MGE", "IN-CGE", "IPC", "IN-fetal", "RG"],
        prob_heatmap_value_col="perturbed_prob",
    )
fate


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "tf" / "erbb4" / "fate" / "perturbation_sweep_heatmap_prob.png")
